# 🔍 Bước 3: Diagnostic Analytics
Tìm nguyên nhân tác động đến doanh thu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
CHARTS = '../output/charts'

try:
    df = pd.read_pickle('../output/df_clean.pkl')
except:
    df = pd.read_csv('../data/shopping_trends.csv')
    for col in ['Gender','Category','Season','Subscription Status']:
        df[col] = df[col].astype('category')
    bins = [0,18,25,35,45,55,100]
    df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=['<18','18-25','26-35','36-45','46-55','56+'])

print("✅ Dữ liệu sẵn sàng:", df.shape)


## 3.1 Correlation Analysis

In [ ]:
df_enc = df.copy()
for col in ['Gender','Season','Category','Subscription Status','Discount Applied','Promo Code Used','Frequency of Purchases']:
    df_enc[col+'_enc'] = df_enc[col].astype(str).astype('category').cat.codes

num_cols = ['Age','Purchase Amount (USD)','Review Rating','Previous Purchases',
            'Gender_enc','Season_enc','Category_enc','Subscription Status_enc',
            'Discount Applied_enc','Promo Code Used_enc']
num_cols = [c for c in num_cols if c in df_enc.columns]

corr = df_enc[num_cols].corr().round(3)
rename = {'Purchase Amount (USD)':'Purchase Amt','Review Rating':'Review',
          'Previous Purchases':'Prev Purchases','Gender_enc':'Gender',
          'Season_enc':'Season','Category_enc':'Category',
          'Subscription Status_enc':'Subscribed','Discount Applied_enc':'Discount',
          'Promo Code Used_enc':'Promo'}
corr.rename(index=rename, columns=rename, inplace=True)

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Heatmap Tương Quan giữa các Biến', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTương quan với Purchase Amount:")
print(corr['Purchase Amt'].drop('Purchase Amt').sort_values(key=abs, ascending=False))


## 3.2 Pivot Table Analysis

In [ ]:
# Pivot: Gender x Season
pivot1 = df.pivot_table(values='Purchase Amount (USD)',
                        index='Gender', columns='Season',
                        aggfunc='mean', observed=True).round(2)
pivot1.index   = pivot1.index.astype(str)
pivot1.columns = pivot1.columns.astype(str)
print("Doanh thu TB: Gender × Season")
display(pivot1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(pivot1, annot=True, fmt='.1f', cmap='Blues', ax=axes[0], linewidths=0.5)
axes[0].set_title('Doanh Thu TB: Gender × Season')

# Pivot: Age Group x Category
pivot2 = df.pivot_table(values='Purchase Amount (USD)',
                        index='Age Group', columns='Category',
                        aggfunc='mean', observed=True).round(2)
pivot2.index   = pivot2.index.astype(str)
pivot2.columns = pivot2.columns.astype(str)
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1], linewidths=0.5)
axes[1].set_title('Doanh Thu TB: Age Group × Category')

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_08_pivot.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.3 T-Test: Subscriber vs Non-Subscriber

In [ ]:
sub_yes = df[df['Subscription Status'].astype(str)=='Yes']['Purchase Amount (USD)'].dropna()
sub_no  = df[df['Subscription Status'].astype(str)=='No']['Purchase Amount (USD)'].dropna()

t_stat, p_value = stats.ttest_ind(sub_yes, sub_no, equal_var=False)

print("=" * 45)
print("  WELCH'S T-TEST: Subscriber vs Non-Subscriber")
print("=" * 45)
print(f"  Mean Subscriber    : ${sub_yes.mean():.2f}")
print(f"  Mean Non-Subscriber: ${sub_no.mean():.2f}")
print(f"  T-statistic        : {t_stat:.4f}")
print(f"  P-value            : {p_value:.4f}")
print(f"  Kết luận           : {'✅ Có' if p_value<0.05 else '❌ Không có'} sự khác biệt có ý nghĩa (α=0.05)")


## 3.4 ANOVA: Doanh thu theo Mùa & Danh mục

In [ ]:
# ANOVA theo Mùa
groups_season = [grp['Purchase Amount (USD)'].values
                 for _, grp in df.groupby('Season', observed=True)]
f_season, p_season = stats.f_oneway(*groups_season)

# ANOVA theo Danh mục
groups_cat = [grp['Purchase Amount (USD)'].values
              for _, grp in df.groupby('Category', observed=True)]
f_cat, p_cat = stats.f_oneway(*groups_cat)

print("=" * 45)
print("  ONE-WAY ANOVA")
print("=" * 45)
print(f"  [Mùa]     F={f_season:.3f}  p={p_season:.4f}  → {'✅ Có ý nghĩa' if p_season<0.05 else '❌ Không ý nghĩa'}")
print(f"  [Danh mục] F={f_cat:.3f}  p={p_cat:.4f}  → {'✅ Có ý nghĩa' if p_cat<0.05 else '❌ Không ý nghĩa'}")

# Boxplot so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_plot = df.copy()
df_plot['Season'] = df_plot['Season'].astype(str)
df_plot['Subscription Status'] = df_plot['Subscription Status'].astype(str)

sns.boxplot(data=df_plot, x='Season', y='Purchase Amount (USD)',
            hue='Subscription Status', palette='Set2', ax=axes[0])
axes[0].set_title('Purchase Amount: Season × Subscription')

df_plot['Category'] = df_plot['Category'].astype(str)
sns.boxplot(data=df_plot, x='Category', y='Purchase Amount (USD)',
            palette='pastel', ax=axes[1])
axes[1].set_title('Purchase Amount: theo Danh Mục')

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_09_anova_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
